In [1]:
%pip install pennylane pennylane-lightning qiskit qiskit-aer matplotlib scipy pyqsp --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pennylane as qml
import numpy as np

# 1. Define a Hermitian matrix (must be square, e.g., 2x2 for 1 data qubit)
H_raw = np.array([[0.4, 0.2], 
                  [0.2, 0.6]])

# Verify it is Hermitian
assert np.allclose(H_raw, H_raw.conj().T), "Matrix must be Hermitian!"

# 2. Scale the matrix so its norm is <= 1 - delta (e.g., delta = 0.1)
norm = np.linalg.norm(H_raw, ord=2)
delta = 0.1
H = H_raw / (norm + delta) 

print(f"Norm of H: {np.linalg.norm(H, ord=2):.4f} (Must be < 1.0)")

# 3. Create the (1, a, 0) Block Encoding V_H
# A 2x2 matrix requires 1 data qubit.
# Let's say we want to simulate an inefficient encoding using a=2 ancilla qubits.
# Total wires needed = 2 (ancilla) + 1 (data) = 3 wires.
a = 2
data_qubits = 1
total_wires = a + data_qubits

dev = qml.device("default.qubit", wires=total_wires)

@qml.qnode(dev)
def get_V_H():
    # qml.BlockEncode creates an exact (error=0), unscaled (alpha=1) 
    # unitary embedding of H using the available ancilla wires.
    qml.BlockEncode(H, wires=range(total_wires))
    
    # We return the matrix representation to verify it
    return qml.state()

# Extract the unitary matrix V_H to use in the rest of Algorithm 1
V_H_matrix = qml.matrix(get_V_H)()
print(V_H_matrix)
print("\nShape of the resulting unitary V_H:", V_H_matrix.shape)

Norm of H: 0.8786 (Must be < 1.0)
[[ 0.48566865  0.24283432  0.81364644 -0.20769431  0.          0.
   0.          0.        ]
 [ 0.24283432  0.72850297 -0.20769431  0.60595213  0.          0.
   0.          0.        ]
 [ 0.81364644 -0.20769431 -0.48566865 -0.24283432  0.          0.
   0.          0.        ]
 [-0.20769431  0.60595213 -0.24283432 -0.72850297  0.          0.
   0.          0.        ]
 [ 0.          0.          0.          0.          1.          0.
   0.          0.        ]
 [ 0.          0.          0.          0.          0.          1.
   0.          0.        ]
 [ 0.          0.          0.          0.          0.          0.
   1.          0.        ]
 [ 0.          0.          0.          0.          0.          0.
   0.          1.        ]]

Shape of the resulting unitary V_H: (8, 8)
